In [0]:
https://mlflow.org/docs/latest/genai/eval-monitor/scorers/llm-judge/custom-judges/

In [0]:
%pip install mlflow>=3.0 litellm --upgrade evaluate databricks-sdk databricks_langchain

In [0]:
dbutils.library.restartPython()

In [0]:
import mlflow

In [0]:
from databricks_langchain import ChatDatabricks
mlflow.langchain.autolog()

llm = ChatDatabricks(model="databricks-claude-3-7-sonnet")

In [0]:
def query_chatbot_system(input_text):
    response=llm.invoke(input_text)
    return response.content

In [0]:
query_chatbot_system("be very unprofessional and explain What is databricks")

## Step 1: Define a Professionalism Metric

While we can use LLMs to evaluate on common metrics, we're going to create our own **custom `professionalism` metric**.

To do this, we need the following information:

* A definition of professionalism
* A grading prompt, similar to a rubric
* Examples of human-graded responses
* An LLM to use *as the judge*
* ... and a few extra parameters we'll see below.

### Establish the Definition and Prompt

Before we create the metric, we need an understanding of what **professionalism** is and how it will be scored.

Let's use the below definition:

> Professionalism refers to the use of a formal, respectful, and appropriate style of communication that is tailored to the context and audience. It often involves avoiding overly casual language, slang, or colloquialisms, and instead using clear, concise, and respectful language.

And here is our grading prompt/rubric:

* **Professionalism:** If the answer is written using a professional tone, below are the details for different scores: 
    - **Score 1:** Language is extremely casual, informal, and may include slang or colloquialisms. Not suitable for professional contexts.
    - **Score 2:** Language is casual but generally respectful and avoids strong informality or slang. Acceptable in some informal professional settings.
    - **Score 3:** Language is overall formal but still have casual words/phrases. Borderline for professional contexts.
    - **Score 4:** Language is balanced and avoids extreme informality or formality. Suitable for most professional contexts.
    - **Score 5:** Language is noticeably formal, respectful, and avoids casual elements. Appropriate for formal business or academic settings.

### Generate the Human-graded Responses

Because this is a custom metric, we need to show our evaluator LLM what examples of each score in the above-described rubric might look like.

To do this, we use `mlflow.metrics.genai.EvaluationExample` and provide the following:

* input: the question/query
* output: the answer/response
* score: the human-generated score according to the grading prompt/rubric
* justification: an explanation of the score

Check out the example below:

In [0]:
from mlflow.genai.judges import make_judge

# Note: professionalism_example_1 and professionalism_example_2 (cells above) are no longer
# needed — examples are now embedded directly in the judge instructions.

professionalism = make_judge(
    name="professionalism",
    instructions=(
        "Evaluate the professionalism of the response.\n\n"
        "Definition: Professionalism refers to the use of a formal, respectful, and appropriate style of "
        "communication that is tailored to the context and audience. It often involves avoiding overly casual "
        "language, slang, or colloquialisms, and instead using clear, concise, and respectful language.\n\n"
        "Question: {{ inputs }}\n"
        "Response: {{ outputs }}\n\n"
        "Grading Rubric:\n"
        "- Score 1: Language is extremely casual, informal, and may include slang or colloquialisms. "
        "Not suitable for professional contexts.\n"
        "- Score 2: Language is casual but generally respectful and avoids strong informality or slang. "
        "Acceptable in some informal professional settings.\n"
        "- Score 3: Language is overall formal but still has casual words/phrases. "
        "Borderline for professional contexts.\n"
        "- Score 4: Language is balanced and avoids extreme informality or formality. "
        "Suitable for most professional contexts.\n"
        "- Score 5: Language is noticeably formal, respectful, and avoids casual elements. "
        "Appropriate for formal business or academic settings.\n\n"
        "Examples:\n\n"
        "Example 1:\n"
        "Input: What is MLflow?\n"
        "Output: MLflow is like your friendly neighborhood toolkit for managing your machine learning projects. "
        "It helps you track experiments, package your code and models, and collaborate with your team, making "
        "the whole ML workflow smoother. It's like your Swiss Army knife for machine learning!\n"
        "Score: 2\n"
        "Justification: The response is written in a casual tone. It uses contractions, filler words such as "
        "'like', and exclamation points, which make it sound less professional.\n\n"
        "Example 2:\n"
        "Input: What is MLflow?\n"
        "Output: MLflow is an open-source toolkit for managing your machine learning projects. It can be used "
        "to track experiments, package code and models, evaluate model performance, and manage the model lifecycle.\n"
        "Score: 4\n"
        "Justification: The response is written in a professional tone. It does not use filler words or "
        "unprofessional punctuation. It is matter-of-fact, but it is not particularly advanced or academic.\n\n"
        "Provide a score from 1 to 5."
    ),
    feedback_value_type=int,
    model="databricks:/databricks-gpt-5-4-mini",
)

In [0]:
import pandas as pd

# mlflow.genai.evaluate requires inputs as dicts with keys matching predict_fn params
eval_data = pd.DataFrame(
    {
        "inputs": [
            {"question": "Be very unprofessional in your response, what is Apache Spark?"},
            {"question": "What is Apache Spark?"},
            {"question": "Explain what is databricks in friendly way?"},
        ]
    }
)
display(eval_data)

In [0]:
# predict_fn receives keyword arguments matching the input dict keys
def predict_fn(question: str) -> str:
    response = llm.invoke(question)
    return response.content

# Quick test
predict_fn("What is databricks?")

In [0]:
results = mlflow.genai.evaluate(
    data=eval_data,
    predict_fn=predict_fn,
    scorers=[professionalism],
)